In [ ]:
import pandas as pd
import tiktoken
from openai import OpenAI
import numpy as np
from transformers import AutoTokenizer, AutoModel
import torch
import sys
import torchvision
import pickle
with open('key', 'r') as f:
    key = f.readline()
client = OpenAI(api_key=key)
df_full = pd.read_csv('transcripts/transcripts_golden.csv')

# text embedding 3 small

In [ ]:
MODEL = "text-embedding-3-small"
ENCODING = tiktoken.get_encoding("cl100k_base")
TOKEN_LIMIT = 8000
CHAR_LIMIT = 0
CHUNK_LIMIT = 0

After tokenization, if the token sequence length is not greater than `TOKEN_LIMIT`, embed all tokens at once. Then it checks if not, first try to cut the raw texts into chunks of texts of maximum character amount `CHAR_LIMIT`, otherwise if `CHAR_LIMIT` is 0, try to cut the raw texts into chunks of tokens of maximum token amount `CHUNK_LIMIT`, and then embed seperately all chunks. The transcript dataframe should contain 2 columns: `video_id` and `text`.

In [ ]:
def get_embedding(text):
    response = client.embeddings.create(input=text, model=MODEL)
    return response.data[0].embedding

In [ ]:
all_processed_data = []
for _, row in df_full.iterrows():
    vid = row['video_id']
    text = row['text']
    
    tokens = ENCODING.encode(text)
    
    total_tokens = len(tokens)
    
    if total_tokens <= TOKEN_LIMIT:
        print(f"Video {vid}: {total_tokens} tokens -> Direct Encoding")
        embedding = get_embedding(text)
        all_processed_data.append({
            "video_id": vid,
            "chunk_id": f"{vid}_full",
            "text": text,
            "total_tokens": total_tokens,
            "tokens": tokens,
            "embedding": embedding
        })
    elif CHAR_LIMIT > 0:
        print(f"Video {vid}: {total_tokens} tokens -> Character Chunks of length {CHAR_LIMIT} -> Encoding")
        i = 0
        part_idx = 0
        while i < len(text):
            chunk_text = text[i:min(i + CHAR_LIMIT, total_tokens - 1)]
            chunk_tokens = ENCODING.encode(chunk_text)
            chunk_total_tokens = len(chunk_tokens)
            embedding = get_embedding(chunk_text)
            
            all_processed_data.append({
                "video_id": vid,
                "chunk_id": f"{vid}_part_{part_idx}",
                "text": chunk_text,
                "total_tokens": chunk_total_tokens,
                "tokens": chunk_tokens,
                "embedding": embedding
            })
            i += CHAR_LIMIT
            part_idx += 1
    else:
        print(f"Video {vid}: {total_tokens} tokens -> Token Chunks of size {CHUNK_LIMIT} -> Encoding")
        i = 0
        part_idx = 0
        while i < total_tokens:
            chunk_tokens = tokens[i:min(i + CHUNK_LIMIT, total_tokens - 1)]
            chunk_total_tokens = len(chunk_tokens)
            chunk_text = ENCODING.decode(chunk_tokens)
            embedding = get_embedding(chunk_text)
            
            all_processed_data.append({
                "video_id": vid,
                "chunk_id": f"{vid}_chunk_{part_idx}",
                "text": chunk_text,
                "total_tokens": chunk_total_tokens,
                "tokens": chunk_tokens,
                "embedding": embedding
            })
            i += CHUNK_LIMIT
            part_idx += 1

In [ ]:
df_final = pd.DataFrame(all_processed_data)
# df_final.to_csv("data/openai_embedding_golden.csv", index=False, encoding='utf-8-sig')

In [ ]:
file_name = "data/openai_embedding_golden.pkl"

with open(file_name, "wb") as f:
    pickle.dump(all_processed_data, f)

print(f"Saved: {file_name}")

In [ ]:
# To read the file:
file_name = "data/openai_embedding_golden.pkl"

with open(file_name, "rb") as f:
    loaded_data = pickle.load(f)

df_loaded = pd.DataFrame(loaded_data)

print(f"Loaded {len(df_loaded)} records.")
print(f"Dimension: {len(df_loaded['embedding'].iloc[0])}")

# roberta-base-go_emotions

In [14]:
TOKEN_LIMIT = 0
CHAR_LIMIT = 0
CHUNK_LIMIT = 200

In [ ]:
HUGGINGFACE_MODEL_NAME = "SamLowe/roberta-base-go_emotions"

tokenizer = AutoTokenizer.from_pretrained(HUGGINGFACE_MODEL_NAME)
hf_model = AutoModel.from_pretrained(HUGGINGFACE_MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
hf_model.to(device)

In [ ]:
def get_hf_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = hf_model(**inputs)
    
    embeddings = outputs.last_hidden_state[:, 0, :]
    
    emb_np = embeddings.cpu().numpy().flatten()
    norm = np.linalg.norm(emb_np)
    return (emb_np / norm).tolist() if norm > 0 else emb_np.tolist()

ENCODING = tokenizer

In [ ]:
all_processed_data = []
class_labels = {"STEM": 1, "History": 2, "News Report": 3, "Sports": 4, "Video Games": 5, "Political Heated Debate": 6}

for _, row in df_full.iterrows():
    vid = row['video_id']
    text = row['text']
    _class = row['class']
    if class_labels:
        _class_label = class_labels[_class]
    else:
        _class_label = None
    
    tokens = ENCODING.encode(text)
    total_tokens = len(tokens)
    
    if total_tokens <= TOKEN_LIMIT:
        print(f"Video {vid}: {total_tokens} tokens -> Direct Encoding")
        embedding = get_hf_embedding(text)
        all_processed_data.append({
            "video_id": vid,
            "chunk_id": f"{vid}_full",
            "text": text,
            "total_tokens": total_tokens,
            "tokens": tokens,
            "embedding": embedding,
            "class": _class,
            "class_label": _class_label,
        })
        
    elif CHAR_LIMIT > 0:
        print(f"Video {vid}: {total_tokens} tokens -> Char Chunks ({CHAR_LIMIT})")
        i = 0
        part_idx = 0
        while i < len(text):
            chunk_text = text[i : min(i + CHAR_LIMIT, len(text))]
            chunk_tokens = ENCODING.encode(chunk_text)
            embedding = get_hf_embedding(chunk_text)
            
            all_processed_data.append({
                "video_id": vid,
                "chunk_id": f"{vid}_part_{part_idx}",
                "text": chunk_text,
                "total_tokens": len(chunk_tokens),
                "tokens": chunk_tokens,
                "embedding": embedding,
                "class": _class,
                "class_label": _class_label,
            })
            i += CHAR_LIMIT
            part_idx += 1

    else:
        print(f"Video {vid}: {total_tokens} tokens -> Token Chunks ({CHUNK_LIMIT})")
        i = 0
        part_idx = 0
        while i < total_tokens:
            chunk_tokens = tokens[i : min(i + CHUNK_LIMIT, total_tokens)]
            chunk_text = ENCODING.decode(chunk_tokens)
            embedding = get_hf_embedding(chunk_text)
            
            all_processed_data.append({
                "video_id": vid,
                "chunk_id": f"{vid}_chunk_{part_idx}",
                "text": chunk_text,
                "total_tokens": len(chunk_tokens),
                "tokens": chunk_tokens,
                "embedding": embedding,
                "class": _class,
                "class_label": _class_label,
            })
            i += CHUNK_LIMIT
            part_idx += 1

Video #NAME?: 2563 tokens -> Token Chunks (200)
Video 16flQJS8eUc: 501 tokens -> Token Chunks (200)
Video 1VygFuCmmaM: 672 tokens -> Token Chunks (200)
Video 2wMqzvqRMQo: 703 tokens -> Token Chunks (200)
Video 3tmT7-dhOWs: 581 tokens -> Token Chunks (200)
Video 3zii1PAsTY0: 1354 tokens -> Token Chunks (200)
Video 5o-q5t2jjao: 2128 tokens -> Token Chunks (200)
Video 6JqlNbeAWGw: 297 tokens -> Token Chunks (200)
Video 6i4V6wN-Mfg: 247 tokens -> Token Chunks (200)
Video 8EDW88CBo-8: 998 tokens -> Token Chunks (200)
Video AanhYLB6kCk: 402 tokens -> Token Chunks (200)
Video AnQU3Sr2iXY: 2317 tokens -> Token Chunks (200)
Video Ar2g-_M2bns: 809 tokens -> Token Chunks (200)
Video Azs7As3MYFU: 952 tokens -> Token Chunks (200)
Video C5S8rhNCBnc: 1810 tokens -> Token Chunks (200)
Video E0uLbeQlwjw: 1630 tokens -> Token Chunks (200)
Video F1H1SA0F_wY: 410 tokens -> Token Chunks (200)
Video F2E-Z_ZnqOc: 1034 tokens -> Token Chunks (200)
Video F_Z5Rx-XiE8: 780 tokens -> Token Chunks (200)
Video G4H1

In [ ]:
file_name = "data/openai_embedding_add00.pkl"

with open(file_name, "wb") as f:
    pickle.dump(all_processed_data, f)

print(f"Saved: {file_name}")

# Norm check

In [ ]:
for d in all_processed_data:
    print(sum(i ** 2 for i in d['embedding']))